# BEARS Colab Pro Runbook

Run this notebook from top to bottom. You should only need to edit the first config cell if your GitHub repo or Google Drive paths differ.

What this notebook includes:

- Colab-safe dependency setup that keeps Colab's CUDA PyTorch instead of downgrading to the original 2022 pins.
- Path handling for Google Drive folders with spaces, such as `PES - Semester 4`.
- PyTorch 2.6+ compatibility for old dataset artifacts saved with NumPy arrays.
- Smoke tests for HalfMNIST training, optional HalfMNIST checkpoint evaluation, optional MiniKandinsky, and BDD-OIA preprocessing/training.
- Optional full practical BDD-OIA run using ResNet50 ImageNet features from `lastframe.zip`.
- Result archiving and copy-back to Google Drive.

Note: the BDD full run here uses ResNet50 replacement features. It is useful for running the pipeline, but it is not the exact paper-faithful Faster-RCNN/CBM `bdd_2048.zip` feature setup.

In [ ]:
#@title 1. Configuration
# Edit only this cell when paths or run choices change.

REPO_URL = "https://github.com/Vishu235/bears.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
REPO_DIR = "/content/bears"  #@param {type:"string"}

DRIVE_DATA_DIR = "/content/drive/MyDrive/PES - Semester 4/bears_data"  #@param {type:"string"}
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/PES - Semester 4/bears_results"  #@param {type:"string"}

LASTFRAME_ZIP = f"{DRIVE_DATA_DIR}/lastframe.zip"
HALFMNIST_CKPT_IN_DRIVE = f"{DRIVE_DATA_DIR}/halfmnist-mnistdpl-dis-None-end.pt"

# Smoke-test switches. Defaults are safe to run on a T4.
RUN_HALF_OOD_EVAL = False  #@param {type:"boolean"}
RUN_MINIKAND_SMOKE = True  #@param {type:"boolean"}
RUN_BDD_SMOKE = True  #@param {type:"boolean"}

# Full BDD is off by default because it can take a while and consumes Colab quota.
RUN_FULL_BDD = False  #@param {type:"boolean"}
FULL_BDD_EPOCHS = 30  #@param {type:"integer"}
FULL_BDD_BATCH_SIZE = 256  #@param {type:"integer"}
FULL_BDD_FEATURE_BATCH_SIZE = 64  #@param {type:"integer"}
FULL_BDD_W_ENTROPY = 1.0  #@param {type:"number"}

# BDD smoke settings.
BDD_SMOKE_LIMIT_PER_SPLIT = 8  #@param {type:"integer"}
BDD_SMOKE_FEATURE_BATCH_SIZE = 8  #@param {type:"integer"}

print("Configuration ready.")

In [ ]:
#@title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 3. Clone or update repository
import os
import subprocess
from pathlib import Path

repo_dir = Path(REPO_DIR)
if repo_dir.exists() and (repo_dir / ".git").exists():
    os.chdir(repo_dir)
    subprocess.run(["git", "fetch", "origin"], check=True)
    subprocess.run(["git", "checkout", BRANCH], check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
elif repo_dir.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repo. Remove it or choose another REPO_DIR.")
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
    os.chdir(repo_dir)

print("Repo ready:", Path.cwd())
subprocess.run(["git", "log", "--oneline", "-3"], check=False)

In [ ]:
#@title 4. Install Colab-safe dependencies
import os
import subprocess

os.chdir(REPO_DIR)
subprocess.run(["python", "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "requirements.colab.txt"], check=True)
print("Dependencies installed.")

In [ ]:
#@title 5. Diagnostics
import os
import subprocess

os.chdir(REPO_DIR)
subprocess.run(["python", "colab_runner.py", "--job", "diagnostics"], check=True)

In [ ]:
#@title 6. Check data files and copy optional checkpoint
import shutil
from pathlib import Path

repo_dir = Path(REPO_DIR)
drive_data_dir = Path(DRIVE_DATA_DIR)
lastframe_zip = Path(LASTFRAME_ZIP)
drive_ckpt = Path(HALFMNIST_CKPT_IN_DRIVE)
repo_ckpt = repo_dir / "XOR_MNIST" / "data" / "ckpts" / "halfmnist-mnistdpl-dis-None-end.pt"

print("Drive data dir:", drive_data_dir)
print("lastframe.zip:", lastframe_zip, "exists=", lastframe_zip.exists())
print("Drive HalfMNIST ckpt:", drive_ckpt, "exists=", drive_ckpt.exists())

repo_ckpt.parent.mkdir(parents=True, exist_ok=True)
if drive_ckpt.exists():
    shutil.copy2(drive_ckpt, repo_ckpt)
    print("Copied HalfMNIST checkpoint to", repo_ckpt)
elif repo_ckpt.exists():
    print("HalfMNIST checkpoint already exists in repo workspace:", repo_ckpt)
else:
    print("HalfMNIST checkpoint not found. Checkpoint evaluation will be skipped.")

kand_zip = repo_dir / "XOR_MNIST" / "data" / "kand-3k.zip"
print("MiniKandinsky zip:", kand_zip, "exists=", kand_zip.exists())

In [ ]:
#@title 7. Helper used by all smoke/full jobs
import os
import shlex
import subprocess
from pathlib import Path

def run_job(job, *, required=True, extra_args=None):
    args = [
        "python", "colab_runner.py",
        "--job", job,
        "--lastframe-zip", LASTFRAME_ZIP,
    ]
    if extra_args:
        args.extend(str(item) for item in extra_args)

    print("\n>>> Running:", " ".join(shlex.quote(arg) for arg in args), flush=True)
    completed = subprocess.run(args, cwd=REPO_DIR, check=False)
    if completed.returncode:
        message = f"Job {job} failed with exit code {completed.returncode}. Scroll above for the BEARS traceback."
        if required:
            raise SystemExit(message)
        print("SKIPPED/FAILED:", message)
    else:
        print(f"<<< Completed {job}")
    return completed.returncode

print("Helper ready.")

## Smoke Test Suite

These jobs verify the project is runnable on Colab before any long experiment starts.

- `halfmnist_smoke`: trains HalfMNIST for one epoch.
- `halfmnist_eval`: evaluates the uploaded HalfMNIST checkpoint when present.
- `minikand_smoke`: runs one MiniKandinsky epoch when enabled.
- `bdd_preprocess_smoke`: converts 8 samples per BDD split using untrained ResNet50 features for speed.
- `bdd_train_smoke`: trains BDD for one epoch on the tiny converted smoke set.

In [ ]:
#@title 8. Run HalfMNIST smoke and optional checkpoint evaluation
from pathlib import Path

repo_ckpt = Path(REPO_DIR) / "XOR_MNIST" / "data" / "ckpts" / "halfmnist-mnistdpl-dis-None-end.pt"

run_job("halfmnist_smoke", extra_args=["--epochs", "1", "--batch-size", "64"])

if repo_ckpt.exists():
    run_job("halfmnist_eval", extra_args=["--eval-type", "frequentist"])
    if RUN_HALF_OOD_EVAL:
        run_job("halfmnist_eval", extra_args=["--eval-type", "frequentist", "--use-ood"])
else:
    print("Skipping halfmnist_eval because checkpoint is missing:", repo_ckpt)

In [ ]:
#@title 9. Run MiniKandinsky smoke when available
from pathlib import Path

kand_zip = Path(REPO_DIR) / "XOR_MNIST" / "data" / "kand-3k.zip"
if RUN_MINIKAND_SMOKE and kand_zip.exists():
    run_job("minikand_smoke", extra_args=["--epochs", "1", "--batch-size", "64"])
elif RUN_MINIKAND_SMOKE:
    print("Skipping MiniKandinsky smoke because kand-3k.zip is missing:", kand_zip)
else:
    print("MiniKandinsky smoke disabled by config.")

In [ ]:
#@title 10. Run BDD-OIA smoke when lastframe.zip is available
from pathlib import Path

if RUN_BDD_SMOKE and Path(LASTFRAME_ZIP).exists():
    run_job(
        "bdd_preprocess_smoke",
        extra_args=[
            "--feature-weights", "none",
            "--feature-batch-size", str(BDD_SMOKE_FEATURE_BATCH_SIZE),
            "--limit-per-split", str(BDD_SMOKE_LIMIT_PER_SPLIT),
        ],
    )
    run_job("bdd_train_smoke")
elif RUN_BDD_SMOKE:
    print("Skipping BDD smoke because lastframe.zip is missing:", LASTFRAME_ZIP)
else:
    print("BDD smoke disabled by config.")

## Optional Full Practical BDD Run

This section only runs when `RUN_FULL_BDD = True` in the config cell. It creates ResNet50 ImageNet features for the full `lastframe.zip` dataset, then trains the BDD model.

Suggested T4 settings:

- `FULL_BDD_FEATURE_BATCH_SIZE = 64`, reduce to `32` if feature extraction runs out of memory.
- `FULL_BDD_BATCH_SIZE = 256`, reduce to `128` if training runs out of memory.

In [ ]:
#@title 11. Optional full practical BDD run
from pathlib import Path

if RUN_FULL_BDD:
    if not Path(LASTFRAME_ZIP).exists():
        raise FileNotFoundError(f"RUN_FULL_BDD=True but missing {LASTFRAME_ZIP}")
    run_job(
        "bdd_preprocess_full",
        extra_args=[
            "--feature-weights", "imagenet",
            "--feature-batch-size", str(FULL_BDD_FEATURE_BATCH_SIZE),
        ],
    )
    run_job(
        "bdd_train_full",
        extra_args=[
            "--epochs", str(FULL_BDD_EPOCHS),
            "--bdd-batch-size", str(FULL_BDD_BATCH_SIZE),
            "--w-entropy", str(FULL_BDD_W_ENTROPY),
        ],
    )
else:
    print("Full BDD run is disabled. Set RUN_FULL_BDD=True in the first cell when smoke tests have passed.")

In [ ]:
#@title 12. Archive results
run_job("archive_results")

from pathlib import Path
for path in sorted((Path(REPO_DIR) / "colab_outputs").glob("bears_results_*.zip")):
    print(path, path.stat().st_size, "bytes")

In [ ]:
#@title 13. Copy latest archive to Drive
import shutil
from pathlib import Path

result_dir = Path(DRIVE_RESULTS_DIR)
result_dir.mkdir(parents=True, exist_ok=True)
archives = sorted((Path(REPO_DIR) / "colab_outputs").glob("bears_results_*.zip"), key=lambda p: p.stat().st_mtime)
if not archives:
    raise FileNotFoundError("No result archive found in colab_outputs.")
latest = archives[-1]
target = result_dir / latest.name
shutil.copy2(latest, target)
print("Copied", latest, "to", target)